# Знакомство с библиотекой `transformers` и `gradio`

Классы `AutoTokenizer` и `AutoModelForCausalLM` позволяет загрузить чекпоинты языковой модели и выполнить генерацию текста.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import gradio

C:\Users\giezz\PycharmProjects\m_nlp_course_vyatsu\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


В качестве модели для экспериментов возьмите модель `Qwen/Qwen3-0.6B` с huggingface. Познакомьтесь с описанием модели и её использованием [ссылка](https://huggingface.co/Qwen/Qwen3-0.6B). Веса загружайте в типе `torch.bfloat16` для экономии памяти GPU. Не забудьте в интерфейсе colab подключить GPU.

In [2]:
model_name_or_path = "Qwen/Qwen3-0.6B"
model = AutoModelForCausalLM.from_pretrained(model_name_or_path, dtype=torch.bfloat16, device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

Входной текст для генерации может быть представлен в режиме диалога в виде списка объектов с полями `role` и `content`. Значения поля `role` может принимать значения `system`, `user`, `assistent`, что соответсвует системному, пользовательскому промптам и ответу модели. У каждой модели специальные токены и собственный формат, приведение к которому происходит с помощью метода `apply_chat_template()`

In [3]:
chat = [
  {"role": "system", "content": "Отвечай на русском языке"},
  {"role": "user", "content": "Что ты думаешь о законах робототехники?"},
  {"role": "assistant", "content": "Законы робототехники — это нарастающая область правовой регуляции, которая стремится адаптироваться к быстрому развитию технологий в области искусственного интеллекта (ИИ) и автоматизации."},
  {"role": "user", "content": "А какой второй закон?"}
]
text = tokenizer.apply_chat_template(chat, tokenize=False,
                                       add_generation_prompt=True,
                                       return_tensors="pt"
)
print(text)

<|im_start|>system
Отвечай на русском языке<|im_end|>
<|im_start|>user
Что ты думаешь о законах робототехники?<|im_end|>
<|im_start|>assistant
Законы робототехники — это нарастающая область правовой регуляции, которая стремится адаптироваться к быстрому развитию технологий в области искусственного интеллекта (ИИ) и автоматизации.<|im_end|>
<|im_start|>user
А какой второй закон?<|im_end|>
<|im_start|>assistant



Генерация выполняется с помощью метода generate(). Генерируемый текст большой (с рассуждениями), поэтому раскройте поле вывода.

In [4]:
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)

In [5]:
all_generated_text = tokenizer.decode(generated_ids[0])
print(all_generated_text)

<|im_start|>system
Отвечай на русском языке<|im_end|>
<|im_start|>user
Что ты думаешь о законах робототехники?<|im_end|>
<|im_start|>assistant
Законы робототехники — это нарастающая область правовой регуляции, которая стремится адаптироваться к быстрому развитию технологий в области искусственного интеллекта (ИИ) и автоматизации.<|im_end|>
<|im_start|>user
А какой второй закон?<|im_end|>
<|im_start|>assistant
<think>
Хорошо, пользователь спрашивает о втором законах робототехники. Нужно ответить, но я должен уточнить, что в робототехнике обычно говорят о законодательстве и правовых нормах, а не о конкретных правилах. Возможно, он хотел узнать о конкретных законах, но в контексте робототехники это обычно относится к регулированию. Надо подчеркнуть, что законы робототехники связаны с правовой стороной, а не с конкретными действиями. Проверю, не путается ли пользователь. Ответ должен быть понятным и соответствовать контексту.
</think>

В робототехнике законы и правовые нормы играют важную 

Изучите формат ответа модели и распарсите ответ, например следующим образом.

In [6]:
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
print(f"Рассуждения: {thinking_content}")
print(f"Ответ: {content}")

Рассуждения: <think>
Хорошо, пользователь спрашивает о втором законах робототехники. Нужно ответить, но я должен уточнить, что в робототехнике обычно говорят о законодательстве и правовых нормах, а не о конкретных правилах. Возможно, он хотел узнать о конкретных законах, но в контексте робототехники это обычно относится к регулированию. Надо подчеркнуть, что законы робототехники связаны с правовой стороной, а не с конкретными действиями. Проверю, не путается ли пользователь. Ответ должен быть понятным и соответствовать контексту.
</think>
Ответ: В робототехнике законы и правовые нормы играют важную роль, связывая с развитием технологий. Однако, в контексте правовой регуляции, а не конкретных правовых актов, законы робототехники обычно относятся к законодательству, правовым нормам и правовым практикам. Если вы имеете в виду конкретные законы, которые регулируют робототехнику, то они могут быть связаны с законодательством стран, включая регулирование ИИ и автоматизации.


Используя библиотеку `gradio` можно добавить графический интерфейс. Например, для чатбота будет полезен компонент `ChatInterface`.

**Задание.** Напишите чатбот с графическим интрфейсом и поддержкой истории при генерации ответа. Глубину истории можно задавать, например, через компонент слайдер в интерфейсе.

In [13]:
import gradio as gr

def predict(message, history, history_depth):
    # Ensure history_depth is an integer, default to 0 if None
    history_depth = int(history_depth) if history_depth is not None else 0

    chat = [
        {"role": "system", "content": "Отвечай на русском языке"}
    ]

    effective_history = history[-history_depth:] if history_depth > 0 else []

    for human, assistant in effective_history:
        chat.append({"role": "user", "content": human})
        chat.append({"role": "assistant", "content": assistant})

    # Add the current user message
    chat.append({"role": "user", "content": message})

    # Apply chat template to get the formatted string
    formatted_chat_string = tokenizer.apply_chat_template(chat,
                                                         tokenize=False,
                                                         add_generation_prompt=True)

    # Prepare model inputs from the string
    model_inputs = tokenizer([formatted_chat_string], return_tensors="pt").to(model.device)

    # Generate response
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=1024, # Limit token generation for responsiveness
        do_sample=True,      # Enable sampling for more varied responses
        temperature=0.7,     # Adjust temperature for creativity
        top_p=0.9            # Adjust top_p for diversity
    )

    # Decode and parse output
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    # The token 151668 corresponds to </think> in Qwen.
    # We try to find it to separate 'thinking' part from 'content' based on previous examples.
    try:
        # Find the index of the last occurrence of 151668 (</think>)
        # If not found, `index` remains 0, and `content` will be the entire output.
        index_from_end = output_ids[::-1].index(151668)
        index = len(output_ids) - index_from_end
    except ValueError:
        index = 0 # If </think> is not found, treat everything after prompt as content

    # Extract content, skipping special tokens
    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

    # For a chatbot, usually we only return the main 'content'
    return content

# Create Gradio interface
with gr.Blocks() as demo:
    gr.Markdown(
        """
        # Qwen Chatbot
        Чатбот на базе модели `Qwen/Qwen3-0.6B` с историей диалога.
        """
    )

    chatbot_interface = gr.ChatInterface(
        fn=predict,
        chatbot=gr.Chatbot(height=400, allow_tags=False),
        textbox=gr.Textbox(placeholder="Введите ваше сообщение...", container=False, scale=7),
        examples=[
            ["Привет!"],
            ["Расскажи про законы робототехники"],
            ["Что такое сингулярность?"]
        ],
        title="Qwen Chatbot с историей",
        description="Интерактивный чатбот, использующий модель Qwen3-0.6B и позволяющий настраивать глубину истории диалога."
    )

    with gr.Accordion("Настройки генерации", open=False):
        history_depth_slider = gr.Slider(
            minimum=0,
            maximum=10,
            step=1,
            value=2, # Default history depth
            label="Глубина истории (количество последних пар 'вопрос-ответ')",
            info="Определяет, сколько последних сообщений из истории диалога будет передано модели для генерации ответа. 0 - только текущий вопрос."
        )
        # Link the slider to the chatbot's predict function as an additional input
        chatbot_interface.additional_inputs = [history_depth_slider]

demo.launch(debug=True, share=True)

C:\Users\giezz\PycharmProjects\m_nlp_course_vyatsu\.venv\Lib\site-packages\gradio\utils.py:1203: UserWarning: Expected 3 arguments for function <function predict at 0x000001940C26B6A0>, received 2.
  warnings.warn(
C:\Users\giezz\PycharmProjects\m_nlp_course_vyatsu\.venv\Lib\site-packages\gradio\utils.py:1207: UserWarning: Expected at least 3 arguments for function <function predict at 0x000001940C26B6A0>, received 2.
  warnings.warn(
C:\Users\giezz\PycharmProjects\m_nlp_course_vyatsu\.venv\Lib\site-packages\gradio\utils.py:1203: UserWarning: Expected 3 arguments for function <function predict at 0x000001940C26B740>, received 2.
  warnings.warn(
C:\Users\giezz\PycharmProjects\m_nlp_course_vyatsu\.venv\Lib\site-packages\gradio\utils.py:1207: UserWarning: Expected at least 3 arguments for function <function predict at 0x000001940C26B740>, received 2.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7860


C:\Users\giezz\PycharmProjects\m_nlp_course_vyatsu\.venv\Lib\site-packages\gradio\helpers.py:1076: UserWarning: Unexpected argument. Filling with None.
  warnings.warn("Unexpected argument. Filling with None.")


* Running on public URL: https://bd628464ac9bc26703.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://bd628464ac9bc26703.gradio.live
